## Purpose of This Notebook

This notebook serves as an exploratory tool for examining the log files produced during the horse behavioural experiments conducted in October and November 2023. It facilitates the experimentation with text parsing techniques on the files before they are imported into a database. The primary objectives are:


> To conduct experiments with regular expressions ([`regex`](https://docs.python.org/3/howto/regex.html)) aimed at extracting pertinent data and fields from the log files.

In [ ]:
# | echo: false

import inspect
import sys
from dataclasses import fields
from pathlib import Path
from pprint import pprint

import duckdb
import pandas as pd
from horse_logic.logfiles import Event, Experiment, Logfile, Logs, Parameter, Trial
from IPython.display import Markdown, display
from loguru import logger

In [ ]:
# | echo: false

def display_class_definition(class_name):
    display(Markdown(f"```{inspect.getsource(class_name)}```"))

In [ ]:
# | echo: false

import itables.options as opt
from itables import init_notebook_mode

init_notebook_mode(all_interactive=True)  # Display dataframes in a more friendly paginated manner
opt.pageLength = 20


In [ ]:
# | echo: false

def set_custom_logger_format():
    # custom_format = "<level>{level: <8}</level> | <cyan>{name}</cyan>:<cyan>{function}</cyan>:<cyan>{line}</cyan> - <level>{message}</level>"
    custom_format_no_func_info = "<level>{level: <8}</level> | <level>{message}</level>"
    logger.remove()    # Remove the default handler before adding a new one
    logger.add(sys.stderr, format=custom_format_no_func_info)

set_custom_logger_format()

In [ ]:
# | echo: false

# Custom notebook analysis functions

def display_sessions_by_subject(df_out, subject_df):
    session_summary = []
    for n_subject, subject_name in enumerate(subject_df["subject_name"]):
        df_ = df_out.drop(columns=["suffix", "subject_name"])[df_out["subject_name"] == subject_name]
        n_session = len(df_)
        session_summary.append((n_subject+1, subject_name.capitalize(), n_session))
        if n_session == 0:
            logger.info(f"Subject {subject_name}: No experiments conducted")
        else:
            df_.loc[:, "time_dff"] = df_["datetime"].diff()
            display(Markdown(f"### {n_subject+1}. {subject_name.capitalize()}: {n_session} session(s)"))
            display(df_)
    return pd.DataFrame(session_summary, columns=["Subject number", "Subject name", "Session count"])

### Load Subject order

In [ ]:
# | echo: false

HORSE_ORDER_XLSX = "Horseorder ALL exp 1.xlsx"
HORSE_ORDER_PATH = Path("../docs/provided") 

HORSE_ORDER_FILEPATH = HORSE_ORDER_PATH / HORSE_ORDER_XLSX

if (HORSE_ORDER_FILEPATH).exists():
    subject_df = pd.read_excel(HORSE_ORDER_FILEPATH)
    logger.info(f"Loaded horse order info from: {HORSE_ORDER_FILEPATH}")
else:
    logger.error(f"Horse order info not found: {HORSE_ORDER_FILEPATH}")

In [ ]:
subject_df.rename({"No": "subject_number", "Horse": "subject_name"}, axis=1, inplace=True)

In [ ]:
subject_df["subject_name"] = subject_df["subject_name"].str.lower()     # ensure lower case names for later lookup

## Log file reconciliation

### Directory information: log file data and outputs

In [ ]:
# | echo: false

LOGFILES_DIR = "../data/results/zips/data_17Jan2020_email_hillydale_equine"

assert Path(LOGFILES_DIR).exists()
logger.info(f"Logfiles dir: {LOGFILES_DIR}")


In [ ]:
# | echo: false

OUTPUT_DIR = "../data/results/notebooks"

assert Path(OUTPUT_DIR).exists()
logger.info(f"Outputs dir: {OUTPUT_DIR}")

In [ ]:
# | echo: false

DATA_DIR = "../data"

assert Path(DATA_DIR).exists()

DATA_DB  = Path(DATA_DIR) / "experiments_October2023.ddb"  # DuckDB name
logger.info(f"Database file: {DATA_DB}")

### Load logfiles from CSV list of included files

In [ ]:
(Path(OUTPUT_DIR) / "all_included_files.csv").exists()

In [ ]:
logs = Logs(path=LOGFILES_DIR)
logs.load_specific_logfiles_from_csv(Path(OUTPUT_DIR) / "all_included_files.csv")

In [ ]:
logs

#### Included log files

In [ ]:
# | echo: false

df_included = pd.DataFrame(logs.logfiles)

`TODO: The list of included files should be read from the previous notebook: logfile-reconciliation.ipynb`

### Explore `regex` parsing of a single example log file

Analysis code is in `logfiles.py`

Specific example defined below.

In [ ]:
# | echo: false

logfile_eg = f"{logs.path}/Experiment_2023-11-17T05:11:53.655051_clover_nan_RPE-R.log"

In [ ]:
# | echo: false

# Load example log file into Logfile class for processing
log_eg = Logfile(logfile_eg)

In [ ]:
# | echo: false

print(f"Example log file for regex parsing: {log_eg.file_name}")

### Data structures

These are created as `dataclasses` in Python and `TABLES` in SQL.

#### Experiment table

In [ ]:
# | echo: false

display_class_definition(Experiment)

#### Parse experiment details from log filename

In [ ]:
# | echo: false

pprint(log_eg.parse_filename_components())

In [ ]:
# | echo: false

log_eg.parse_comments()

#### Parameters table

In [ ]:
# | echo: false

display_class_definition(Parameter)

#### Parse parameters example

In [ ]:
# | echo: false

log_eg.parse_parameters()

#### Trials and Events

In [ ]:
# | echo: false

display_class_definition(Trial)

In [ ]:
# | echo: false

display_class_definition(Event)

#### Parsing of trials and events example

In [ ]:
# | echo: false

pprint(log_eg.parse_trials_and_events()[0])

In [ ]:
# | echo: false

n_trial = len(log_eg.parse_trials_and_events()[0])

In [ ]:
# | echo: false
 
df_trial_event = pd.DataFrame(log_eg.parse_trials_and_events()[1])

In [ ]:
df_trial_event

In [ ]:
# | echo: false

def display_events_by_trial_number(df_trial_event):
    n_trial = df_trial_event["TrialNumber"].max()
    for i_trial in range(n_trial):
        df_trial_event.loc[:, "time_diff"] = pd.to_datetime(df_trial_event["EventDateTime"]).diff()
        display(Markdown(f"### Trial #{i_trial+1}"))
        display(df_trial_event[df_trial_event["TrialNumber"] == i_trial+1])

In [ ]:
# | echo: false

df_ = pd.DataFrame(log_eg.parse_trials_and_events()[1])

In [ ]:
# | echo: false

df_

In [ ]:
# | echo: false

# Test extract info for trials

for i_trial in range(df_["TrialNumber"].max()):
    for row in df_[df_["TrialNumber"] == i_trial].itertuples():
        print(i_trial, row[2], row[3])


In [ ]:
# | echo: false

# df_trial_event.to_csv("df_trial_event_eg.csv")

In [ ]:
# | echo: false

display_events_by_trial_number(df_trial_event)

In [ ]:
# | echo: false

pprint(log_eg.parse_filename_components())

### Put data in DuckDB database

In [ ]:
def insert_record(con, record):
    table_name = record.__class__.__name__.lower() + 's'  # Assumes table name is plural of the class name
    columns = ', '.join(field.name for field in fields(record))
    placeholders = ', '.join('?' for _ in fields(record))
    values = tuple(getattr(record, field.name) for field in fields(record))
    query = f'INSERT INTO {table_name} ({columns}) VALUES ({placeholders})'
    try:
        con.execute(query, values)
    except Exception as e:
        print(f"Query: {query}")
        print(f"Values: {values}")

In [ ]:
if DATA_DB.exists():
    DATA_DB.unlink()
    
con = duckdb.connect(database=str(DATA_DB))

In [ ]:
def create_tables_from_sql_file(con, file_path):
    with open(file_path, 'r') as file:
        sql_commands = file.read()
    display(Markdown(f"\n```\n{sql_commands}\n```"))
    con.execute(sql_commands)

In [ ]:
create_tables_from_sql_file(con, '../sql/create_experiment_tables_ddb.sql')

In [ ]:
def load_parse_experiment_log(logfile, path="../data/results/zips/data_17Jan2020_email_hillydale_equine/"):
     return Logfile(f"{path}/{logfile}").parse_filename_components(), pd.DataFrame(Logfile(f"{path}/{logfile}").parse_trials_and_events()[1])


exp_details, trial_details = load_parse_experiment_log("Experiment_2023-10-02T21:28:43.621970_bonnie_2_RPE-R.log")

In [ ]:
trial_details

In [ ]:
trial_numbers = trial_details["TrialNumber"].unique()

In [ ]:
trial_start_time = trial_details.iloc[0]["EventDateTime"]
trial_end_time = trial_details.iloc[-1]["EventDateTime"]


In [ ]:
trial_start_time, trial_end_time

In [ ]:
extra_excludes = [
    "Experiment_2023-10-03T14:14:32.928978_bonnie_10_RPE-A.log",
    "Experiment_2023-10-02T21:28:43.621970_bonnie_2_RPE-R.log",
    "Experiment_2023-10-03T14:55:06.781765_bonnie_15_RPE-ER.log",
    "Experiment_2023-10-03T15:46:19.429509_bonnie_16_RPE-ER.log",
    "Experiment_2023-10-04T11:15:57.929926_bonnie_36_RPE-A.log"
]

In [ ]:
logfiles_to_process = [logfile.file_name for logfile in logs.logfiles if logfile.file_name not in extra_excludes]

In [ ]:
len(logfiles_to_process)

In [ ]:
# Initialise the unique (integer) id for each Trial (TrialID) and each Event (EventID)

trial_id = 0
event_id = 0

for exp_id, logfile in enumerate(sorted(logfiles_to_process)): # Loop over Experiments - starting at exp_id = 0 (ExperimentID)
    logger.info(f"{exp_id}: {logs.path}/{logfile}")
    logf = Logfile(f"{logs.path}/{logfile}")
    exp_details = logf.parse_filename_components()
    print(exp_details)
    exp_details.ExperimentID = exp_id
    exp_details.SubjectNumber = int(subject_df.loc[subject_df["subject_name"] == exp_details.SubjectName, "subject_number"].iloc[0])   #logger.info(f"Experiment: {exp_details}")

    print(logf.file_name)
    print(logf.parse_trials_and_events())

    insert_record(con, exp_details) # Add each Experiment

    # Loop over Trials
    trial_details_df = pd.DataFrame(Logfile(f"{logs.path}/{logfile}").parse_trials_and_events()[1])

    n_trial = trial_details_df["TrialNumber"].max()

    print(trial_details_df)
    print(n_trial)

    for trial_number in range(n_trial):
        trial_details_df.loc[:, "time_diff"] = pd.to_datetime(trial_details_df["EventDateTime"]).diff()

        trial_start_time = trial_details.iloc[0]["EventDateTime"]   # Start tone of the first event in the trial
        trial_end_time = trial_details.iloc[-1]["EventDateTime"]    # Last event time in the trial

        #display(Markdown(f"### Trial #{i_trial+1}"))
        #display(df_trial_event[df_trial_event["TrialNumber"] == i_trial+1])

        insert_record(con, Trial(trial_id, exp_id, trial_number, trial_start_time, trial_end_time)) 

        trial_id += 1
        
    #     # Loop over Events

    #     # TODO - Unpack events from DataFrame?



In [ ]:
con.sql("SHOW TABLES;")

In [ ]:
experiments_df = con.sql("SELECT * FROM Experiments").df()

In [ ]:
experiments_df

In [ ]:
con.sql("SELECT DISTINCT ExperimentType FROM Experiments")

In [ ]:
con.sql("SELECT DISTINCT SubjectName FROM Experiments ORDER BY SubjectName")

In [ ]:
con.sql("SELECT * FROM Trials")

In [ ]:
con.sql("SELECT * FROM Events")